## What Does a SWE Salary Actually Buy?

Purchasing power comparison: how many Big Macs, monthly rents, and median local
wages does a software engineer's annual salary represent in each country?

Sources: The Economist Big Mac Index 2023; Numbeo cost-of-living 2023.

In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import plotly.express as px

# Embedded cost-of-living proxies (local currency, 2023)
COST_DATA = {
    'USA':   {'big_mac_local': 5.58,   'monthly_rent_local': 1500},
    'India': {'big_mac_local': 190,    'monthly_rent_local': 15000},
    'China': {'big_mac_local': 24,     'monthly_rent_local': 3500},
}
country_colors = {'USA': '#2171b5', 'India': '#31a354', 'China': '#e6550d'}

usa = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_usa_data.csv')
india = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_india_data.csv')
china = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_china_data.csv')
df = pd.concat([usa, india, china], ignore_index=True)
mid = df[df['career_stage'] == 'mid'].copy()
swe = mid[mid['role'] == 'software_engineer'].copy()

rows = []
for _, row in swe.iterrows():
    c = row['country']
    salary = row['median_salary_local']
    costs = COST_DATA[c]
    rows.append({
        'country': c,
        'annual_salary_local': salary,
        'big_macs_per_year': int(salary / costs['big_mac_local']),
        'months_rent_per_year': round(salary / costs['monthly_rent_local'], 1),
    })
pp = pd.DataFrame(rows)
print(pp.to_string(index=False))

country  annual_salary_local  big_macs_per_year  months_rent_per_year
    USA               130160              23326                  86.8
  India              1200000               6315                  80.0
  China               200000               8333                  57.1


In [2]:
fig = px.bar(
    pp, x='country', y='big_macs_per_year', color='country',
    color_discrete_map=country_colors,
    title='SWE Annual Salary: How Many Big Macs? (2023, local currency)<br>'
          '<sup>Proxy for local purchasing power — more = better real purchasing power</sup>',
    labels={'big_macs_per_year': 'Big Macs per year', 'country': 'Country'},
    text='big_macs_per_year',
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.show()

In [3]:
fig2 = px.bar(
    pp, x='country', y='months_rent_per_year', color='country',
    color_discrete_map=country_colors,
    title='SWE Annual Salary: How Many Months of (Metro) Rent? (2023, local currency)',
    labels={'months_rent_per_year': 'Months of rent', 'country': 'Country'},
    text='months_rent_per_year',
)
fig2.update_traces(texttemplate='%{text:.1f} months', textposition='outside')
fig2.show()

In [4]:
farm = mid[mid['role'] == 'farm_worker'][['country', 'median_salary_local']].rename(
    columns={'median_salary_local': 'farm_salary'}
)
compare = pp[['country', 'annual_salary_local']].merge(farm, on='country')
compare['swe_bm'] = compare.apply(lambda r: int(r['annual_salary_local'] / COST_DATA[r['country']]['big_mac_local']), axis=1)
compare['farm_bm'] = compare.apply(lambda r: int(r['farm_salary'] / COST_DATA[r['country']]['big_mac_local']), axis=1)
long = pd.melt(compare[['country', 'swe_bm', 'farm_bm']], id_vars='country',
               value_vars=['swe_bm', 'farm_bm'],
               var_name='role', value_name='big_macs')
long['role'] = long['role'].map({'swe_bm': 'Software Engineer', 'farm_bm': 'Farm Worker'})

fig3 = px.bar(
    long, x='country', y='big_macs', color='role',
    barmode='group',
    color_discrete_sequence=['#2171b5', '#8c6d31'],
    title='Annual Purchasing Power: SWE vs Farm Worker (Big Macs per year, 2023)',
    labels={'big_macs': 'Big Macs per year (local currency)', 'country': 'Country', 'role': 'Role'},
)
fig3.show()